In [1]:
!nvidia-smi

Mon Aug 24 19:06:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q "colpali-engine>=0.3.15" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 6.7 MB/s eta 0:00:00


In [3]:
import torch
import colpali_engine
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

PyTorch: 2.11.0+cu128
Transformers: 5.15.0
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [4]:
import torch
from colpali_engine.models import ColQwen3_5, ColQwen3_5Processor

MODEL_ID = "tencent/EVIE-Preview-4.5B"

torch.cuda.empty_cache()

print("Loading EVIE-Preview-4.5B...")
print("This may take several minutes on the first run.")

model = ColQwen3_5.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
    attn_implementation="eager",
    low_cpu_mem_usage=True,
).eval()


def enable_bidirectional_attention(model):
    for cfg in (
        model.config,
        getattr(model.config, "text_config", None),
    ):
        if cfg is not None:
            cfg.is_causal = False

    for module in model.modules():
        if module.__class__.__name__ in (
            "Qwen3_5Attention",
            "Qwen3Attention",
        ):
            if hasattr(module, "is_causal"):
                module.is_causal = False


enable_bidirectional_attention(model)

print("\nLoading processor...")

processor = ColQwen3_5Processor.from_pretrained(
    MODEL_ID
)

print("\nEVIE loaded successfully!")

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU memory reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

print("Model dtype:", next(model.parameters()).dtype)
print("Device:", next(model.parameters()).device)

Loading EVIE-Preview-4.5B...
This may take several minutes on the first run.


config.json:   0%|          | 0.00/2.84k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 9.08GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/725 [00:00<?, ?it/s]

...

chat_template.jinja:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

tokenizer.json: downloading bytes:           |  0.00B            

[ERROR] `min_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.
[ERROR] `max_frames` is part of Qwen3VLVideoProcessorInitKwargs, but not documented. Make sure to add it to the docstring of the function in /usr/local/lib/python3.13/dist-packages/transformers/models/qwen3_vl/video_processing_qwen3_vl.py.

EVIE loaded successfully!
GPU memory allocated: 8.46 GB
GPU memory reserved: 8.47 GB
Model dtype: torch.float16
Device: cuda:0


In [5]:
from huggingface_hub import hf_hub_download
from PIL import Image
import torch

# Download 4 example document pages
page_paths = [
    hf_hub_download(
        "sentence-transformers/example-documents",
        f"doc{i}.jpg",
        repo_type="dataset"
    )
    for i in range(1, 5)
]

images = [
    Image.open(path).convert("RGB")
    for path in page_paths
]

queries = [
    "What is the variable represented on the y-axis of the graph?",
    "Total outlay is maximum in which year?",
]

print("Preparing images and queries...")

image_batch = processor.process_images(images).to(model.device)
query_batch = processor.process_queries(queries).to(model.device)

print("Creating EVIE embeddings...")

with torch.inference_mode():

    image_embeddings = model(**image_batch)

    # Important before processing queries
    model.rope_deltas = None

    query_embeddings = model(**query_batch)

scores = processor.score(
    query_embeddings,
    image_embeddings
)

print("\nScores:")
print(scores)

print("\nBest page per query:")
print(scores.argmax(dim=1))

doc1.jpg: reconstructing file:   0%|          |  0.00B /  126kB            

doc1.jpg: downloading bytes:           |  0.00B            

doc2.jpg: reconstructing file:   0%|          |  0.00B /  922kB            

doc2.jpg: downloading bytes:           |  0.00B            

doc3.jpg: reconstructing file:   0%|          |  0.00B /  534kB            

doc3.jpg: downloading bytes:           |  0.00B            

doc4.jpg: reconstructing file:   0%|          |  0.00B /  509kB            

doc4.jpg: downloading bytes:           |  0.00B            

Preparing images and queries...
Creating EVIE embeddings...

Scores:
tensor([[17.3594, 10.7969,  7.9375,  7.3008],
        [ 6.5469, 13.4219,  6.1992,  6.1211]])

Best page per query:
tensor([0, 1])


In [6]:
!pip install -q datasets

In [7]:
from datasets import load_dataset

DATASET_NAME = "vidore/vidore_v3_finance_en"
NUM_PAGES = 20

print("Loading first 20 finance document pages...")

corpus = load_dataset(
    DATASET_NAME,
    "corpus",
    split="test",
    streaming=True,
)

pages = []

for row in corpus:
    pages.append(
        {
            "corpus_id": row["corpus_id"],
            "doc_id": row["doc_id"],
            "page_number": row["page_number_in_doc"],
            "image": row["image"].convert("RGB"),
        }
    )

    if len(pages) >= NUM_PAGES:
        break

print("\nPages loaded:", len(pages))

print("\nCorpus IDs:")
print([page["corpus_id"] for page in pages])

# Confirm our known ground-truth page is included
ground_truth = [
    page
    for page in pages
    if page["corpus_id"] == 10
]

print("\nGround truth check:")

if ground_truth:
    print("Corpus ID:", ground_truth[0]["corpus_id"])
    print("Document:", ground_truth[0]["doc_id"])
    print("PDF page:", ground_truth[0]["page_number"])
else:
    print("Corpus ID 10 NOT FOUND")

Loading first 20 finance document pages...


README.md:   0%|          | 0.00/9.49k [00:00<?, ?B/s]


Pages loaded: 20

Corpus IDs:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

Ground truth check:
Corpus ID: 10
Document: jpmorgan_chase_2024
PDF page: 107


In [8]:
import torch
import gc

QUERY = (
    "Did JPMorganChase execute more than half of its planned "
    "$30 billion stock repurchase program by year-end?"
)

GROUND_TRUTH_CORPUS_ID = 10

print("Embedding 20 PDF pages with EVIE...")

page_embeddings = []

# Process ONE page at a time to protect T4 memory
for i, page in enumerate(pages, start=1):

    image_batch = processor.process_images(
        [page["image"]]
    ).to(model.device)

    with torch.inference_mode():
        embedding = model(**image_batch)

    # Store embedding on CPU instead of keeping all on GPU
    page_embeddings.append(
        embedding.cpu()
    )

    del image_batch
    del embedding

    torch.cuda.empty_cache()
    gc.collect()

    print(
        f"Embedded {i}/{len(pages)} "
        f"- Corpus ID {page['corpus_id']}"
    )


# --------------------------------------------------
# Embed the query
# --------------------------------------------------

print("\nEmbedding query...")

model.rope_deltas = None

query_batch = processor.process_queries(
    [QUERY]
).to(model.device)

with torch.inference_mode():
    query_embedding = model(**query_batch)


# --------------------------------------------------
# Score every page
# --------------------------------------------------

print("\nScoring pages...")

scores = []

for page_embedding in page_embeddings:

    page_embedding_gpu = page_embedding.to(model.device)

    with torch.inference_mode():

        score = processor.score(
            query_embedding,
            page_embedding_gpu
        )

    scores.append(
        float(score.item())
    )

    del page_embedding_gpu

    torch.cuda.empty_cache()


# --------------------------------------------------
# Rank pages
# --------------------------------------------------

ranked = sorted(
    zip(pages, scores),
    key=lambda x: x[1],
    reverse=True
)


print("\n" + "=" * 70)
print("EVIE TOP 5 RESULTS")
print("=" * 70)


for rank, (page, score) in enumerate(
    ranked[:5],
    start=1
):

    print(f"\nRANK #{rank}")
    print(f"EVIE score: {score:.4f}")
    print(f"Corpus ID: {page['corpus_id']}")
    print(f"Document: {page['doc_id']}")
    print(f"PDF page: {page['page_number']}")


# --------------------------------------------------
# Ground-truth rank
# --------------------------------------------------

ground_truth_rank = None

for rank, (page, score) in enumerate(
    ranked,
    start=1
):

    if page["corpus_id"] == GROUND_TRUTH_CORPUS_ID:
        ground_truth_rank = rank
        ground_truth_score = score
        break


print("\n" + "=" * 70)
print("GROUND TRUTH CHECK")
print("=" * 70)

print("Expected Corpus ID:", GROUND_TRUTH_CORPUS_ID)
print("EVIE rank:", ground_truth_rank)
print("EVIE score:", round(ground_truth_score, 4))

Embedding 20 PDF pages with EVIE...
Embedded 1/20 - Corpus ID 0
Embedded 2/20 - Corpus ID 1
Embedded 3/20 - Corpus ID 2
Embedded 4/20 - Corpus ID 3
Embedded 5/20 - Corpus ID 4
Embedded 6/20 - Corpus ID 5
Embedded 7/20 - Corpus ID 6
Embedded 8/20 - Corpus ID 7
Embedded 9/20 - Corpus ID 8
Embedded 10/20 - Corpus ID 9
Embedded 11/20 - Corpus ID 10
Embedded 12/20 - Corpus ID 11
Embedded 13/20 - Corpus ID 12
Embedded 14/20 - Corpus ID 13
Embedded 15/20 - Corpus ID 14
Embedded 16/20 - Corpus ID 15
Embedded 17/20 - Corpus ID 16
Embedded 18/20 - Corpus ID 17
Embedded 19/20 - Corpus ID 18
Embedded 20/20 - Corpus ID 19

Embedding query...

Scoring pages...

EVIE TOP 5 RESULTS

RANK #1
EVIE score: 24.7500
Corpus ID: 10
Document: jpmorgan_chase_2024
PDF page: 107

RANK #2
EVIE score: 18.6406
Corpus ID: 1
Document: jpmorgan_chase_2024
PDF page: 1

RANK #3
EVIE score: 17.8438
Corpus ID: 17
Document: jpmorgan_chase_2024
PDF page: 113

RANK #4
EVIE score: 17.7500
Corpus ID: 19
Document: jpmorgan_chase

In [9]:
from datasets import load_dataset
import torch
import gc

QUERY = (
    "Did JPMorganChase execute more than half of its planned "
    "$30 billion stock repurchase program by year-end?"
)

GROUND_TRUTH_CORPUS_ID = 10
NUM_PAGES = 100

# --------------------------------------------------
# 1. Load first 100 pages
# --------------------------------------------------

print(f"Loading first {NUM_PAGES} finance document pages...")

corpus = load_dataset(
    "vidore/vidore_v3_finance_en",
    "corpus",
    split="test",
    streaming=True,
)

pages_100 = []

for row in corpus:
    pages_100.append(
        {
            "corpus_id": row["corpus_id"],
            "doc_id": row["doc_id"],
            "page_number": row["page_number_in_doc"],
            "image": row["image"].convert("RGB"),
        }
    )

    if len(pages_100) >= NUM_PAGES:
        break

print("Pages loaded:", len(pages_100))

# --------------------------------------------------
# 2. Embed 100 pages one at a time
# --------------------------------------------------

print("\nEmbedding 100 PDF pages with EVIE...")

page_embeddings_100 = []

for i, page in enumerate(pages_100, start=1):

    image_batch = processor.process_images(
        [page["image"]]
    ).to(model.device)

    with torch.inference_mode():
        embedding = model(**image_batch)

    page_embeddings_100.append(embedding.cpu())

    del image_batch
    del embedding

    torch.cuda.empty_cache()
    gc.collect()

    if i % 10 == 0 or i == len(pages_100):
        print(f"Embedded {i}/{len(pages_100)} pages")

# --------------------------------------------------
# 3. Embed query
# --------------------------------------------------

print("\nEmbedding query...")

model.rope_deltas = None

query_batch = processor.process_queries(
    [QUERY]
).to(model.device)

with torch.inference_mode():
    query_embedding = model(**query_batch)

# --------------------------------------------------
# 4. Score pages
# --------------------------------------------------

print("\nScoring pages...")

scores_100 = []

for i, page_embedding in enumerate(page_embeddings_100, start=1):

    page_embedding_gpu = page_embedding.to(model.device)

    with torch.inference_mode():
        score = processor.score(
            query_embedding,
            page_embedding_gpu
        )

    scores_100.append(float(score.item()))

    del page_embedding_gpu
    torch.cuda.empty_cache()

    if i % 20 == 0 or i == len(page_embeddings_100):
        print(f"Scored {i}/{len(page_embeddings_100)} pages")

# --------------------------------------------------
# 5. Rank pages
# --------------------------------------------------

ranked_100 = sorted(
    zip(pages_100, scores_100),
    key=lambda x: x[1],
    reverse=True
)

print("\n" + "=" * 70)
print("EVIE TOP 10 RESULTS (100 PAGES)")
print("=" * 70)

for rank, (page, score) in enumerate(ranked_100[:10], start=1):
    print(f"\nRANK #{rank}")
    print(f"EVIE score: {score:.4f}")
    print(f"Corpus ID: {page['corpus_id']}")
    print(f"Document: {page['doc_id']}")
    print(f"PDF page: {page['page_number']}")

# --------------------------------------------------
# 6. Ground-truth rank
# --------------------------------------------------

ground_truth_rank = None
ground_truth_score = None

for rank, (page, score) in enumerate(ranked_100, start=1):
    if page["corpus_id"] == GROUND_TRUTH_CORPUS_ID:
        ground_truth_rank = rank
        ground_truth_score = score
        break

print("\n" + "=" * 70)
print("GROUND TRUTH CHECK")
print("=" * 70)

print("Expected Corpus ID:", GROUND_TRUTH_CORPUS_ID)
print("EVIE rank:", ground_truth_rank)
print("EVIE score:", round(ground_truth_score, 4))

Loading first 100 finance document pages...
Pages loaded: 100

Embedding 100 PDF pages with EVIE...
Embedded 10/100 pages
Embedded 20/100 pages
Embedded 30/100 pages
Embedded 40/100 pages
Embedded 50/100 pages
Embedded 60/100 pages
Embedded 70/100 pages
Embedded 80/100 pages
Embedded 90/100 pages
Embedded 100/100 pages

Embedding query...

Scoring pages...
Scored 20/100 pages
Scored 40/100 pages
Scored 60/100 pages
Scored 80/100 pages
Scored 100/100 pages

EVIE TOP 10 RESULTS (100 PAGES)

RANK #1
EVIE score: 24.7500
Corpus ID: 10
Document: jpmorgan_chase_2024
PDF page: 107

RANK #2
EVIE score: 21.0625
Corpus ID: 87
Document: jpmorgan_chase_2024
PDF page: 177

RANK #3
EVIE score: 20.9375
Corpus ID: 88
Document: jpmorgan_chase_2024
PDF page: 178

RANK #4
EVIE score: 18.8438
Corpus ID: 86
Document: jpmorgan_chase_2024
PDF page: 176

RANK #5
EVIE score: 18.6406
Corpus ID: 1
Document: jpmorgan_chase_2024
PDF page: 1

RANK #6
EVIE score: 18.2188
Corpus ID: 39
Document: jpmorgan_chase_2024
PD

In [10]:
from datasets import load_dataset

DATASET_NAME = "vidore/vidore_v3_finance_en"
NUM_QUERIES = 10

# --------------------------------------------------
# 1. Get the 100 corpus IDs already indexed by EVIE
# --------------------------------------------------

corpus_ids_100 = [
    page["corpus_id"]
    for page in pages_100
]

print("Indexed pages:", len(corpus_ids_100))


# --------------------------------------------------
# 2. Load qrels
# --------------------------------------------------

print("\nLoading relevance judgments...")

qrels_dataset = load_dataset(
    DATASET_NAME,
    "qrels",
    split="test",
    streaming=True,
)

# query_id -> {corpus_id: relevance_score}
evie_qrels = {}

for row in qrels_dataset:

    query_id = row["query_id"]
    corpus_id = row["corpus_id"]
    score = row["score"]

    # Same rule we used for BGE-M3:
    # only keep relevant pages inside our 100-page corpus
    if corpus_id in corpus_ids_100:

        if query_id not in evie_qrels:
            evie_qrels[query_id] = {}

        evie_qrels[query_id][corpus_id] = score


print(
    "Queries with relevant pages in corpus:",
    len(evie_qrels)
)


# --------------------------------------------------
# 3. Select the same first 10 valid queries
# --------------------------------------------------

print("\nLoading benchmark queries...")

queries_dataset = load_dataset(
    DATASET_NAME,
    "queries",
    split="test",
    streaming=True,
)

evie_queries = []

for row in queries_dataset:

    query_id = row["query_id"]

    if query_id in evie_qrels:

        evie_queries.append(
            {
                "query_id": query_id,
                "query": row["query"],
                "relevance": evie_qrels[query_id],
            }
        )

    if len(evie_queries) >= NUM_QUERIES:
        break


# --------------------------------------------------
# 4. Display exactly what EVIE will be tested on
# --------------------------------------------------

print("\n" + "=" * 70)
print("EVIE BENCHMARK QUERY SET")
print("=" * 70)

for i, item in enumerate(evie_queries, start=1):

    print(f"\n{i}. Query ID: {item['query_id']}")
    print("Question:", item["query"])
    print("Relevant pages:", item["relevance"])


print("\nTotal benchmark queries:", len(evie_queries))

print(
    "\nQuery IDs:",
    [item["query_id"] for item in evie_queries]
)

Indexed pages: 100

Loading relevance judgments...
Queries with relevant pages in corpus: 114

Loading benchmark queries...

EVIE BENCHMARK QUERY SET

1. Query ID: 0
Question: Did JPMorganChase execute more than half of its planned $30 billion stock repurchase program by year-end?
Relevant pages: {10: 2}

2. Query ID: 42
Question: JPMorgan Chase supplementary leverage ratio 2024
Relevant pages: {6: 1, 9: 2}

3. Query ID: 59
Question: Is JPMorgan Chase's 2024 CET1 ratio higher than Bank of America's stress capital buffer percentage?
Relevant pages: {3: 1, 7: 1}

4. Query ID: 60
Question: Extract the firm's Common Equity Tier 1 (CET1) capital ratio as part of its strategic priorities under Basel III requirements.
Relevant pages: {3: 1, 5: 1, 7: 1}

5. Query ID: 63
Question: How might a visual comparison of credit loss allowance trends between the two banks illustrate differences in their approaches to credit risk provisioning?
Relevant pages: {28: 1, 33: 1, 42: 1, 86: 1}

6. Query ID: 67

In [11]:
import torch
import numpy as np
import math
import time
import csv
import gc


# --------------------------------------------------
# Metric helpers
# --------------------------------------------------

def recall_at_k(ranked_ids, relevance, k):
    relevant_ids = set(relevance.keys())
    retrieved_ids = set(ranked_ids[:k])

    hits = len(
        relevant_ids.intersection(retrieved_ids)
    )

    return hits / len(relevant_ids)


def dcg_at_k(ranked_ids, relevance, k):
    dcg = 0.0

    for rank, corpus_id in enumerate(
        ranked_ids[:k],
        start=1
    ):
        rel = relevance.get(corpus_id, 0)

        if rel > 0:
            dcg += (
                (2 ** rel - 1)
                / math.log2(rank + 1)
            )

    return dcg


def ndcg_at_k(ranked_ids, relevance, k):
    actual_dcg = dcg_at_k(
        ranked_ids,
        relevance,
        k
    )

    ideal_relevances = sorted(
        relevance.values(),
        reverse=True
    )

    ideal_dcg = 0.0

    for rank, rel in enumerate(
        ideal_relevances[:k],
        start=1
    ):
        ideal_dcg += (
            (2 ** rel - 1)
            / math.log2(rank + 1)
        )

    if ideal_dcg == 0:
        return 0.0

    return actual_dcg / ideal_dcg


# --------------------------------------------------
# Calculate EVIE embedding index size
# --------------------------------------------------

index_bytes = sum(
    tensor.numel() * tensor.element_size()
    for tensor in page_embeddings_100
)

index_mb = index_bytes / (1024 ** 2)

print(
    f"EVIE embedding index size: "
    f"{index_mb:.2f} MB"
)


# --------------------------------------------------
# Run 10-query benchmark
# --------------------------------------------------

results = []
query_latencies = []

print("\nRunning EVIE 10-query benchmark...")


for i, item in enumerate(evie_queries, start=1):

    query_id = item["query_id"]
    query_text = item["query"]
    relevance = item["relevance"]

    print("\n" + "-" * 70)
    print(f"Query {i}/{len(evie_queries)}")
    print("Query ID:", query_id)
    print("Question:", query_text)

    # --------------------------------------------------
    # Start latency timer
    # Includes query embedding + retrieval scoring
    # --------------------------------------------------

    start_time = time.perf_counter()

    model.rope_deltas = None

    query_batch = processor.process_queries(
        [query_text]
    ).to(model.device)

    with torch.inference_mode():
        query_embedding = model(**query_batch)

    # --------------------------------------------------
    # Score query against all 100 EVIE page embeddings
    # --------------------------------------------------

    scores = []

    for page_embedding in page_embeddings_100:

        page_embedding_gpu = page_embedding.to(
            model.device
        )

        with torch.inference_mode():
            score = processor.score(
                query_embedding,
                page_embedding_gpu
            )

        scores.append(
            float(score.item())
        )

        del page_embedding_gpu

    latency = time.perf_counter() - start_time

    query_latencies.append(latency)

    # --------------------------------------------------
    # Rank pages
    # --------------------------------------------------

    ranked_indices = np.argsort(
        scores
    )[::-1]

    ranked_ids = [
        pages_100[index]["corpus_id"]
        for index in ranked_indices
    ]

    # --------------------------------------------------
    # Find first relevant page
    # --------------------------------------------------

    first_relevant_rank = None

    for rank, corpus_id in enumerate(
        ranked_ids,
        start=1
    ):
        if corpus_id in relevance:
            first_relevant_rank = rank
            break

    # --------------------------------------------------
    # Metrics
    # --------------------------------------------------

    hit_at_1 = int(
        first_relevant_rank <= 1
    )

    hit_at_5 = int(
        first_relevant_rank <= 5
    )

    hit_at_10 = int(
        first_relevant_rank <= 10
    )

    recall_5 = recall_at_k(
        ranked_ids,
        relevance,
        5
    )

    recall_10 = recall_at_k(
        ranked_ids,
        relevance,
        10
    )

    ndcg_5 = ndcg_at_k(
        ranked_ids,
        relevance,
        5
    )

    ndcg_10 = ndcg_at_k(
        ranked_ids,
        relevance,
        10
    )

    reciprocal_rank = (
        1 / first_relevant_rank
    )

    # --------------------------------------------------
    # Display query result
    # --------------------------------------------------

    print(
        "Relevant pages:",
        relevance
    )

    print(
        "First relevant rank:",
        first_relevant_rank
    )

    print(
        "Top 5 corpus IDs:",
        ranked_ids[:5]
    )

    print(
        f"Hit@1={hit_at_1} | "
        f"Hit@5={hit_at_5} | "
        f"Hit@10={hit_at_10}"
    )

    print(
        f"Recall@5={recall_5:.3f} | "
        f"Recall@10={recall_10:.3f}"
    )

    print(
        f"nDCG@5={ndcg_5:.3f} | "
        f"nDCG@10={ndcg_10:.3f}"
    )

    print(
        f"Latency={latency:.4f}s"
    )

    results.append(
        {
            "query_id": query_id,
            "query": query_text,
            "first_relevant_rank": first_relevant_rank,
            "hit_at_1": hit_at_1,
            "hit_at_5": hit_at_5,
            "hit_at_10": hit_at_10,
            "recall_at_5": recall_5,
            "recall_at_10": recall_10,
            "ndcg_at_5": ndcg_5,
            "ndcg_at_10": ndcg_10,
            "reciprocal_rank": reciprocal_rank,
            "query_latency_seconds": latency,
        }
    )

    del query_batch
    del query_embedding

    torch.cuda.empty_cache()
    gc.collect()


# --------------------------------------------------
# Aggregate metrics
# --------------------------------------------------

hit_1 = np.mean(
    [r["hit_at_1"] for r in results]
)

hit_5 = np.mean(
    [r["hit_at_5"] for r in results]
)

hit_10 = np.mean(
    [r["hit_at_10"] for r in results]
)

recall_5 = np.mean(
    [r["recall_at_5"] for r in results]
)

recall_10 = np.mean(
    [r["recall_at_10"] for r in results]
)

ndcg_5 = np.mean(
    [r["ndcg_at_5"] for r in results]
)

ndcg_10 = np.mean(
    [r["ndcg_at_10"] for r in results]
)

mrr = np.mean(
    [r["reciprocal_rank"] for r in results]
)

avg_latency = np.mean(
    query_latencies
)


# --------------------------------------------------
# Final results
# --------------------------------------------------

print("\n" + "=" * 70)
print("EVIE-Preview-4.5B BENCHMARK RESULTS")
print("=" * 70)

print(
    f"Pages indexed:    {len(pages_100)}"
)

print(
    f"Queries:          {len(results)}"
)

print("\nRetrieval quality")

print(
    f"Hit@1:            {hit_1:.3f}"
)

print(
    f"Hit@5:            {hit_5:.3f}"
)

print(
    f"Hit@10:           {hit_10:.3f}"
)

print(
    f"Recall@5:         {recall_5:.3f}"
)

print(
    f"Recall@10:        {recall_10:.3f}"
)

print(
    f"nDCG@5:           {ndcg_5:.3f}"
)

print(
    f"nDCG@10:          {ndcg_10:.3f}"
)

print(
    f"MRR:              {mrr:.3f}"
)

print("\nPerformance")

print(
    f"Avg query latency: "
    f"{avg_latency:.4f} seconds"
)

print(
    f"Embedding size:    "
    f"{index_mb:.2f} MB"
)


# --------------------------------------------------
# Save detailed results in Colab
# --------------------------------------------------

OUTPUT_FILE = (
    "/content/evie_10query_results.csv"
)

with open(
    OUTPUT_FILE,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=results[0].keys()
    )

    writer.writeheader()
    writer.writerows(results)


print(
    "\nDetailed results saved to:",
    OUTPUT_FILE
)

EVIE embedding index size: 18.43 MB

Running EVIE 10-query benchmark...

----------------------------------------------------------------------
Query 1/10
Query ID: 0
Question: Did JPMorganChase execute more than half of its planned $30 billion stock repurchase program by year-end?
Relevant pages: {10: 2}
First relevant rank: 1
Top 5 corpus IDs: [10, 87, 88, 86, 1]
Hit@1=1 | Hit@5=1 | Hit@10=1
Recall@5=1.000 | Recall@10=1.000
nDCG@5=1.000 | nDCG@10=1.000
Latency=0.5870s

----------------------------------------------------------------------
Query 2/10
Query ID: 42
Question: JPMorgan Chase supplementary leverage ratio 2024
Relevant pages: {6: 1, 9: 2}
First relevant rank: 1
Top 5 corpus IDs: [9, 6, 7, 16, 3]
Hit@1=1 | Hit@5=1 | Hit@10=1
Recall@5=1.000 | Recall@10=1.000
nDCG@5=1.000 | nDCG@10=1.000
Latency=1.0167s

----------------------------------------------------------------------
Query 3/10
Query ID: 59
Question: Is JPMorgan Chase's 2024 CET1 ratio higher than Bank of America's stre

In [12]:
from google.colab import files

files.download("/content/evie_10query_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>